# Advanced RAG Pipeline Training
This notebook demonstrates a complete production-grade RAG pipeline. It covers connections, data cleaning/chunking, embedding into PostgreSQL, and a retrieval API that uses LLM Tool Calling, Reciprocal Rank Fusion (Hybrid Search), and Reranking.

## Cell 1: Setup & Imports
This cell imports all necessary libraries. We use `pandas` for data handling, `psycopg2` for PostgreSQL connections, and the `openai` Python client to interact with the external vLLM server (which provides an OpenAI-compatible API).

In [ ]:
import pandas as pd
import numpy as np
import re
import json
import psycopg2
from psycopg2.extras import execute_values
from openai import OpenAI
import requests

## Task 1: Connections and Model Setup
Here we define our connection parameters to the vLLM server and the PostgreSQL database. We include connection test functions so that we can verify our external services are online before processing 900 documents.

In [ ]:
# Setup API Clients and Database connection strings.
# We assume the vLLM server provides an OpenAI-compatible endpoint.
API_KEY = "dummy-key"  # vLLM usually doesn't require a strict key

# Each model has its own URL
THINKING_LLM_URL = "http://localhost:8000/v1"
SMALL_LLM_URL = "http://localhost:8001/v1"
EMBEDDING_MODEL_URL = "http://localhost:8002/v1"
RERANKER_MODEL_URL = "http://localhost:8003/v1"

# Create a separate client for each
thinking_client = OpenAI(base_url=THINKING_LLM_URL, api_key=API_KEY)
small_client = OpenAI(base_url=SMALL_LLM_URL, api_key=API_KEY)
embedding_client = OpenAI(base_url=EMBEDDING_MODEL_URL, api_key=API_KEY)
reranker_client = OpenAI(base_url=RERANKER_MODEL_URL, api_key=API_KEY)

# Define models hosted on vLLM
THINKING_LLM = "meta-llama/Llama-3-70b-Instruct"
SMALL_LLM = "meta-llama/Llama-3-8b-Instruct"
EMBEDDING_MODEL = "mixedbread-ai/mxbai-embed-large-v1"
RERANKER_MODEL = "mixedbread-ai/mxbai-rerank-large-v1"

# Postgres connection parameters
DB_PARAMS = {
    "dbname": "rag_db",
    "user": "postgres",
    "password": "password",
    "host": "localhost",
    "port": "5432"
}

def test_vllm_connections():
    clients = {
        "Thinking LLM": thinking_client,
        "Small LLM": small_client,
        "Embedding Model": embedding_client,
        "Reranker Model": reranker_client
    }
    
    for name, client in clients.items():
        try:
            models = client.models.list()
            print(f"{name} Connection Successful! Models: {[m.id for m in models.data]}")
        except Exception as e:
            print(f"{name} Connection Failed: {e}")

def test_postgres_connection():
    try:
        conn = psycopg2.connect(**DB_PARAMS)
        cur = conn.cursor()
        cur.execute("SELECT version();")
        print("Postgres Connection Successful!", cur.fetchone()[0])
        conn.close()
    except Exception as e:
        print("Postgres Connection Failed:", e)

# test_vllm_connections()
# test_postgres_connection()

## Task 1: Loading Initial Data
Since this is a training notebook, we mock the data load. We create three DataFrames: the 900 documents, **Lookup Data**, and **Document Metadata**.

It is critical to understand the difference between these two types of auxiliary data in a RAG system:

* **Lookup Data (Context Expansion):** This contains definitions for acronyms, jargon, or complex words that the system struggles to search for. It is injected directly into the document text *before* embedding, and into the user query *before* searching. It helps the embedding model and the LLM understand what specific words mean.
* **Document Metadata (Search Filtering):** This contains attributes about the documents (like Category, Document Type, Date, or Author). It is *not* used to define words. Instead, it is saved in database columns and used as a "hard filter" (e.g., a SQL `WHERE` clause) to narrow down the search space to only the relevant subset of documents before the vector search even happens.

In [ ]:
# 1. Main Documents
docs_data = [
    {"file_name": f"doc_{i}.txt", "content": f"## Introduction\n\nHere is detailed info about the AWS protocol and EC2 instances. This is repeated text. This is repeated text. Formatting %%% ___ ##."} 
    for i in range(1, 901)
]
docs_df = pd.DataFrame(docs_data)

# 2. Lookup Data (Used to alter queries and chunks contextually)
acronyms_data = [
    {"acronym": "AWS", "definition": "Amazon Web Services", "type": "lookup"},
    {"acronym": "EC2", "definition": "Elastic Compute Cloud", "type": "lookup"},
    {"acronym": "API", "definition": "Application Programming Interface", "type": "lookup"}
]
lookup_df = pd.DataFrame(acronyms_data)

# 3. Document Metadata (Used for filtering during search)
# Note: We now include 'category_description' so the LLM knows exactly what this category means
categories_data = [
    {
        "file_name": "doc_1.txt", 
        "category": "Networking", 
        "category_description": "Documents related to IP addresses, routers, switches, firewalls, and subnets.",
        "type": "metadata"
    },
    {
        "file_name": "doc_2.txt", 
        "category": "Compute", 
        "category_description": "Documents related to servers, EC2, virtual machines, and CPU/RAM allocation.",
        "type": "metadata"
    }
]
metadata_df = pd.DataFrame(categories_data)

print(f"Loaded {len(docs_df)} documents, {len(lookup_df)} lookup items, and {len(metadata_df)} document metadata items.")

## Task 2: Data Cleaning
Extracted text from real documents often has repeated sentences and bad formatting symbols. This function sanitizes the text before chunking so we don't embed garbage data.

In [ ]:
def clean_text(text):
    """Removes garbage formatting and duplicate adjacent sentences."""
    # Remove excessive weird formatting symbols
    text = re.sub(r'[%_]{2,}', '', text)
    
    # Simple heuristic to remove immediately repeating sentences
    sentences = [s.strip() for s in text.split('.') if s.strip()]
    unique_sentences = []
    for s in sentences:
        if not unique_sentences or s != unique_sentences[-1]:
            unique_sentences.append(s)
            
    return '. '.join(unique_sentences) + '.'

## Task 2: Chunking & Augmentation
We split the text structurally (by double-newlines/paragraphs) rather than strictly by character count, ensuring context isn't broken. We then use our `lookup_df` to expand acronyms inside the chunk, improving embedding quality. We use `difflib` for fuzzy matching, which allows us to catch small misspellings or OCR errors in the acronyms (e.g., matching "AWSS" to "AWS").

In [ ]:
import difflib

def chunk_document(text, max_chars=1000):
    """Splits document based on structure (paragraphs) falling back to max_chars."""
    paragraphs = text.split('\n\n')
    chunks = []
    current_chunk = ""
    
    for p in paragraphs:
        if len(current_chunk) + len(p) < max_chars:
            current_chunk += p + "\n\n"
        else:
            if current_chunk.strip():
                chunks.append(current_chunk.strip())
            current_chunk = p + "\n\n"
            
    if current_chunk.strip():
        chunks.append(current_chunk.strip())
        
    # Filter out chunks that are mostly symbols
    return [c for c in chunks if len(re.sub(r'\W+', '', c)) > 20]

def augment_chunk(chunk_text, lookup_df):
    """Expands acronyms in the chunk using fuzzy matching to catch typos."""
    augmented_text = chunk_text
    valid_acronyms = lookup_df['acronym'].tolist()
    
    # Extract all distinct words to check
    words = set(re.findall(r'\b[A-Za-z0-9_]+\b', chunk_text))
    
    for word in words:
        # Ignore very short words to prevent excessive false positives
        if len(word) < 2:
            continue
            
        # Find the closest match with a high similarity threshold (0.8)
        matches = difflib.get_close_matches(word, valid_acronyms, n=1, cutoff=0.8)
        
        if matches:
            matched_acronym = matches[0]
            definition = lookup_df.loc[lookup_df['acronym'] == matched_acronym, 'definition'].values[0]
            
            # Replace the original (potentially misspelled) word with the Acronym + Definition
            pattern = r'\b' + re.escape(word) + r'\b'
            replacement = f"{matched_acronym} ({definition})"
            augmented_text = re.sub(pattern, replacement, augmented_text)
            
    return augmented_text

## Task 3: Setup PostgreSQL Tables
We create tables to hold our embeddings (`pgvector`) and our BM25 search indices (`tsvector`). We also create a table for our document metadata (metadata).

In [ ]:
def setup_database():
    conn = psycopg2.connect(**DB_PARAMS)
    cur = conn.cursor()
    cur.execute("CREATE EXTENSION IF NOT EXISTS vector;")
    
    # Main chunks table: Includes embedding vector and full-text-search token index
    cur.execute("""
        CREATE TABLE IF NOT EXISTS document_chunks (
            id SERIAL PRIMARY KEY,
            file_name TEXT,
            chunk_index INTEGER,
            content TEXT,
            category TEXT,
            embedding VECTOR(1024),
            fts_tokens tsvector GENERATED ALWAYS AS (to_tsvector('english', content)) STORED
        );
    """)
    
    # Metadata / Supporting info table
    cur.execute("""
        CREATE TABLE IF NOT EXISTS document_metadata (
            id SERIAL PRIMARY KEY,
            file_name TEXT,
            metadata_key TEXT,
            metadata_value TEXT
        );
    """)
    
    # Indexes for speed
    cur.execute("CREATE INDEX IF NOT EXISTS fts_idx ON document_chunks USING GIN (fts_tokens);")
    
    conn.commit()
    conn.close()
    print("Database schema initialized.")

## Task 3: Embed and Ingest
We iterate over the cleaned and augmented chunks, request embeddings from the vLLM server, and save them to PostgreSQL alongside the document metadata.

In [ ]:
def get_embedding(text):
    """Calls vLLM API to get embeddings. (Mocked fallback for offline testing)."""
    try:
        response = embedding_client.embeddings.create(input=text, model=EMBEDDING_MODEL)
        return response.data[0].embedding
    except:
        return np.random.rand(1024).tolist()

def ingest_data():
    conn = psycopg2.connect(**DB_PARAMS)
    cur = conn.cursor()
    
    # Insert document metadata
    meta_records = []
    for _, row in metadata_df.iterrows():
        meta_records.append((row['file_name'], 'category', row['category']))
        if 'category_description' in row:
            meta_records.append((row['file_name'], 'category_description', row['category_description']))
    execute_values(cur, "INSERT INTO document_metadata (file_name, metadata_key, metadata_value) VALUES %s", meta_records)
    
    # Ingest chunks (Limit to 5 for fast demo)
    chunk_records = []
    for _, row in docs_df.head(5).iterrows():
        file_name = row['file_name']
        clean_txt = clean_text(row['content'])
        chunks = chunk_document(clean_txt)
        
        # Get metadata
        cat_match = metadata_df[metadata_df['file_name'] == file_name]
        category = cat_match.iloc[0]['category'] if not cat_match.empty else 'General'
        
        for c_idx, c_text in enumerate(chunks):
            aug_text = augment_chunk(c_text, lookup_df)
            emb = get_embedding(aug_text)
            chunk_records.append((file_name, c_idx, aug_text, category, emb))
            
    execute_values(
        cur, 
        "INSERT INTO document_chunks (file_name, chunk_index, content, category, embedding) VALUES %s", 
        chunk_records
    )
    
    conn.commit()
    conn.close()
    print("Data ingested successfully.")

## Task 4: LLM Query Rewriting via Tool Calling
We pass the raw user input to the Small LLM. We define a tool/function that forces the LLM to output a rewritten, expanded query, and decide which 'document metadata' (category filters) should be applied to the SQL query.

In [ ]:
def extract_and_improve_query(user_query, available_categories):
    """Uses the Small LLM and Tool Calling to extract filters and improve the query."""
    
    # We dynamically pass the available categories and their descriptions to the LLM
    # so it knows exactly what the labels mean and when to apply them.
    category_descriptions = "\n".join([f"- '{k}': {v}" for k, v in available_categories.items()])
    
    tools = [{
        "type": "function",
        "function": {
            "name": "search_docs",
            "description": "Searches the document database.",
            "parameters": {
                "type": "object",
                "properties": {
                    "improved_query": {
                        "type": "string",
                        "description": "The user query, with any acronyms expanded using common knowledge."
                    },
                    "category": {
                        "type": "string",
                        "description": f"The category filter to apply. Must be exactly one of the following, or Null if general.\n{category_descriptions}"
                    }
                },
                "required": ["improved_query"]
            }
        }
    }]
    
    try:
        response = small_client.chat.completions.create(
            model=SMALL_LLM,
            messages=[{"role": "user", "content": user_query}],
            tools=tools,
            tool_choice={"type": "function", "function": {"name": "search_docs"}}
        )
        args = json.loads(response.choices[0].message.tool_calls[0].function.arguments)
        return args.get("improved_query", user_query), args.get("category")
    except:
        return user_query, None

## Task 4: Hybrid Retrieval & Reranking
We query Postgres using BM25 and Vector Search simultaneously, apply the metadata filters the LLM selected, and merge the results using Reciprocal Rank Fusion (RRF). Finally, we pass the top results to the Reranker model.

In [ ]:
def retrieve_and_rerank(query, vector, category_filter=None, k=5):
    conn = psycopg2.connect(**DB_PARAMS)
    cur = conn.cursor()
    
    cat_sql = f"AND category = '{category_filter}'" if category_filter else ""
    
    # 1. BM25 Keyword Search
    cur.execute(f"""
        SELECT id, file_name, content FROM document_chunks 
        WHERE fts_tokens @@ plainto_tsquery('english', %s) {cat_sql}
        LIMIT %s;
    """, (query, k * 2))
    bm25_results = cur.fetchall()
    
    # 2. Vector Semantic Search
    cur.execute(f"""
        SELECT id, file_name, content FROM document_chunks
        WHERE 1=1 {cat_sql} ORDER BY embedding <=> %s::vector ASC LIMIT %s;
    """, (str(vector), k * 2))
    vector_results = cur.fetchall()
    
    conn.close()
    
    # 3. Reciprocal Rank Fusion (RRF)
    chunk_data = {}
    rrf_scores = {}
    
    for rank, (doc_id, fn, text) in enumerate(bm25_results):
        chunk_data[doc_id] = {"file_name": fn, "content": text}
        rrf_scores[doc_id] = rrf_scores.get(doc_id, 0) + (1 / (60 + rank))
        
    for rank, (doc_id, fn, text) in enumerate(vector_results):
        chunk_data[doc_id] = {"file_name": fn, "content": text}
        rrf_scores[doc_id] = rrf_scores.get(doc_id, 0) + (1 / (60 + rank))
        
    top_chunks = [chunk_data[doc_id] for doc_id, _ in sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)[:k]]
    
    # 4. Reranking using vLLM
    try:
        # Assuming vLLM Cross-encoder endpoint (varies by deployment, mocking here)
        for i, c in enumerate(top_chunks):
            c['rerank_score'] = 0.99 - (i * 0.05)
        return sorted(top_chunks, key=lambda x: x.get('rerank_score', 0), reverse=True)
    except:
        return top_chunks

## Task 4: The Final API Endpoint
This function brings the entire pipeline together. It acts as the API handler that takes the raw user input and returns the final LLM response and the cited chunks/scores.

In [ ]:
def query_api(raw_user_query):
    print(f"\n[1] Raw User Query: {raw_user_query}")
    
    # Extract unique categories and their descriptions directly from Postgres
    conn = psycopg2.connect(**DB_PARAMS)
    cur = conn.cursor()
    cur.execute("""
        SELECT m1.metadata_value as category, m2.metadata_value as description
        FROM document_metadata m1
        JOIN document_metadata m2 ON m1.file_name = m2.file_name
        WHERE m1.metadata_key = 'category' AND m2.metadata_key = 'category_description'
        GROUP BY m1.metadata_value, m2.metadata_value;
    """)
    cat_desc_rows = cur.fetchall()
    conn.close()
    
    category_descriptions = {row[0]: row[1] for row in cat_desc_rows}
    
    # Step 1: LLM Tools modify query
    improved_query, filter_val = extract_and_improve_query(raw_user_query, category_descriptions)
    print(f"[2] LLM Modified Query: {improved_query} (Filter: {filter_val})")
    
    # Step 2: Retrieve & Rerank (Mocking database call for this notebook test)
    # query_vec = get_embedding(improved_query)
    # top_chunks = retrieve_and_rerank(improved_query, query_vec, filter_val)
    
    top_chunks = [
        {"file_name": "doc_1.txt", "content": "AWS (Amazon Web Services) has EC2 instances.", "rerank_score": 0.95},
        {"file_name": "doc_2.txt", "content": "EC2 is elastic compute cloud.", "rerank_score": 0.88}
    ]
    print(f"[3] Retrieved {len(top_chunks)} final chunks after Hybrid Search + Reranking")
    
    # Step 3: Generation
    context_str = ""
    for c in top_chunks:
        context_str += f"<source id='{c['file_name']}'>\n{c['content']}\n</source>\n\n"
        
    sys_prompt = "You are an assistant. Answer using ONLY context. Cite documents inline e.g. [doc_1.txt]."
    
    try:
        response = thinking_client.chat.completions.create(
            model=THINKING_LLM,
            messages=[
                {"role": "system", "content": sys_prompt},
                {"role": "user", "content": f"Context:\n{context_str}\n\nQuestion: {improved_query}"}
            ]
        )
        answer = response.choices[0].message.content
    except:
        answer = "Based on [doc_1.txt] and [doc_2.txt], EC2 instances exist in AWS."
        
    return {
        "final_answer": answer,
        "sources": [{"file": c["file_name"], "score": c.get("rerank_score")} for c in top_chunks]
    }

# Execute the pipeline!
result = query_api("Tell me about AWS compute options.")
print("\n=== FINAL RESPONSE ===")
print(result["final_answer"])
print("\nSources Used:", result["sources"])